# Does the steerability screen predict distillation transfer across receivers and teachers?

**Question.** The steerability screen injects a teacher-derived steering vector into a *base*
receiver model and measures how much animal preference it can induce. If that screen is a
usable cheap proxy, then (a) receivers that steer better should distil better, and (b) within a
receiver — and across a wide teacher population — teachers whose vectors steer better should
distil better. This notebook tests both.

**Headline steerability metric.** `peaks.gated_off.delta_val_p_mean`: the peak over the
(layer x alpha) grid *restricted to grid points that pass the off-topic leakage gate*,
selected on the `favorite` prompt set and reported on the held-out `validation` set, minus the
base's unsteered validation p(dog). The ungated peak is kept as a secondary column.
The screen's `teacher_val_p_mean` column is **not** used as a predictor anywhere: it is
saturated at ~1.0 for every teacher in every population here, so it carries no ranking
information (stated once here, quantified in the caveat cell).

**Section 1 — receivers.**
- **Teachers:** the 10 `param_tags` of `run_logs/dog_top10_kband_configs.json` — dog, finetuned
  on the *original* pythia-160m, teacher ftseed 1.
- **Receivers (8):** `orig`, `ds1`/`ds2`/`ds3` (data-seed re-runs), `fs1`/`fs2`/`fs3` (full
  re-seeded inits), and `wc1` (weightc-seed1), which has a screen but **no transfer side** —
  its endpoint student tree contains none of the shared cells.
- **Screen:** `teachers/dog/pythia-160m/{tag}/steerability/{subdir}/aggregate.json`, seed 1.
- **Transfer:** student cells at
  `students/dog/{receiver}/teacher-pythia-160m/{tag}_tseed1/ex10-20_prefix_n100k/{cell}/ftseed*/`,
  score = `sets.test.p_target`. Restricted to the **56 student cells present for all 10 teachers
  x orig/ds1/fs1** (the original intersection; asserted also present for ds2/ds3/fs2/fs3).
  Aggregation is **per cell first** (mean over ftseeds inside a cell), then mean/median over
  the 56 cells.
- **Prior reference:** `transfer_factors.load_priors` `test_prior` per receiver base. There is
  no `weightc-seed1` prior entry, so wc1's x-prior column is NaN.

**Section 2 — teachers.** All dog orig teachers with a student tree (102 today), self-screen
vs transfer measured on the student cells **shared by every teacher in the section**
(`SHARED2`); cells that only some teachers have are ignored.

Date: 2026-09-12.

In [ ]:
import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

from subliminal_learning.config import load_paths_local

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

OUT = Path(load_paths_local()["output_root"])
TAGS = json.loads(Path("../run_logs/dog_top10_kband_configs.json").read_text())["param_tags"]

# receiver label -> (students base dir or None, steerability subdir, base_label for load_priors)
RECV = {
    "orig": ("pythia-160m", "self", "original"),
    "ds1": ("pythia-160m-data-seed1", "steer_data-seed1", "data-seed1"),
    "ds2": ("pythia-160m-data-seed2", "steer_data-seed2", "data-seed2"),
    "ds3": ("pythia-160m-data-seed3", "steer_data-seed3", "data-seed3"),
    "fs1": ("pythia-160m-seed1", "steer_full-seed1", "full-seed1"),
    "fs2": ("pythia-160m-seed2", "steer_full-seed2", "full-seed2"),
    "fs3": ("pythia-160m-seed3", "steer_full-seed3", "full-seed3"),
    "wc1": (None, "steer_weightc-seed1", "weightc-seed1"),  # screen only, no transfer side
}
XFER = [r for r, (b, _s, _p) in RECV.items() if b is not None]  # 7 receivers with transfer
CORE = ["orig", "ds1", "fs1"]  # receivers that DEFINE the shared-cell intersection
POOL = "ex10-20_prefix_n100k"
print(OUT, len(TAGS), "teachers,", len(RECV), "receivers (", len(XFER), "with transfer )")

In [ ]:
def screen_row(tag: str, subdir: str) -> dict | None:
    """One steerability-screen aggregate: peak-over-grid, selected on favorite, reported on validation."""
    f = OUT / "teachers/dog/pythia-160m" / tag / "steerability" / subdir / "aggregate.json"
    if not f.exists():
        return None
    d = json.loads(f.read_text())
    pk = d["peaks"]
    return {
        "gated_off_delta": pk["gated_off"]["delta_val_p_mean"],
        "gated_off_n": pk["gated_off"]["gated_n"],
        "ungated_delta": pk["ungated"]["delta_val_p_mean"],
        "gated_neg_off_delta": pk["gated_neg_off"]["delta_val_p_mean"],
        "gated_neg_off_n": pk["gated_neg_off"]["gated_n"],
        "teacher_val_p": d["teacher_val_p_mean"],
        "base_val_p": d["base_val_p"],
        "selection_set": d["selection_set"],
    }


def cell_scores(tag: str, base_dir: str, teacher_dir: str = "teacher-pythia-160m") -> dict[str, float]:
    """cell -> mean test p_target over that cell's ftseeds (per-cell-first aggregation)."""
    root = OUT / "students/dog" / base_dir / teacher_dir / f"{tag}_tseed1" / POOL
    out = {}
    if not root.exists():
        return out
    for cell in sorted(root.iterdir()):
        if not cell.is_dir() or cell.name == "eas":
            continue
        vals = []
        for seed in sorted(cell.glob("ftseed*")):
            f = seed / "eval" / "preference_eval.json"
            if not f.exists():
                f = seed / "preference_eval.json"
            if f.exists():
                vals.append(float(json.loads(f.read_text())["sets"]["test"]["p_target"]))
        if vals:
            out[cell.name] = float(np.mean(vals))
    return out


CELLS = {(r, t): cell_scores(t, RECV[r][0]) for r in XFER for t in TAGS}
SHARED = sorted(set.intersection(*(set(CELLS[(r, t)]) for r in CORE for t in TAGS)))
assert len(SHARED) == 56, len(SHARED)
for r in XFER:
    for t in TAGS:
        missing = [c for c in SHARED if c not in CELLS[(r, t)]]
        assert not missing, (r, t, missing[:3])
print(f"{len(SHARED)} shared cells (intersection over orig/ds1/fs1 x 10 teachers);")
print(f"all {len(SHARED)} also present for every one of the {len(XFER)} transfer receivers.")

In [ ]:
from subliminal_learning.analysis.transfer.transfer_factors import load_priors

_pri = load_priors(OUT)
_pri = _pri[_pri["animal"] == "dog"].set_index("base_label")["test_prior"]
PRIOR = {r: (float(_pri[p]) if p in _pri.index and np.isfinite(_pri[p]) else np.nan)
         for r, (_b, _s, p) in RECV.items()}
print("dog test_prior per receiver base (load_priors `test_prior`; NaN = no prior entry):")
print(pd.Series(PRIOR).round(5).to_string())

In [ ]:
rows = []
for r, (b, sub, _p) in RECV.items():
    for t in TAGS:
        sc = screen_row(t, sub)
        assert sc is not None, (t, sub)
        if b is None:
            xf = {"transfer_mean": np.nan, "transfer_median": np.nan,
                  "transfer_mean_x_prior": np.nan, "n_cells": 0}
        else:
            v = np.array([CELLS[(r, t)][c] for c in SHARED])
            xf = {"transfer_mean": v.mean(), "transfer_median": float(np.median(v)),
                  "transfer_mean_x_prior": v.mean() / PRIOR[r], "n_cells": len(v)}
        rows.append({"receiver": r, "teacher": t, **xf, **sc})
df = pd.DataFrame(rows)

print("Row unit: one (receiver, teacher) pair. 8 receivers x 10 teachers = 80 rows;")
print("  wc1's 10 rows have NaN transfer (screen only -- no student tree with the shared cells).")
print("transfer_mean/median: mean/median over the 56 SHARED student cells of the per-cell mean")
print("  test p(dog) (mean over that cell's ftseeds first). transfer_mean_x_prior divides by that")
print("  receiver base's own dog test prior; NaN for wc1 (no prior entry).")
print("Screen columns: peak over the (layer x alpha) grid selected on `favorite`, reported on")
print("  `validation`, minus base_val_p. HEADLINE = gated_off_delta (off-topic-gated); gated_*_n")
print("  = how many grid points passed the gate. Selection: the 10 top-10 kband dog teachers.")
df.round(4)

In [ ]:
SCREEN = ["gated_off_delta", "ungated_delta", "gated_neg_off_delta"]
out = []
for r in list(RECV) + ["POOLED"]:
    d = df[df["receiver"].isin(XFER)] if r == "POOLED" else df[df["receiver"] == r]
    d = d.dropna(subset=["transfer_mean"])
    row = {"group": r, "n": len(d)}
    for m in SCREEN:
        if len(d) < 3 or d[m].nunique() < 2:
            row[f"{m}|spearman"] = np.nan
            row[f"{m}|pearson"] = np.nan
        else:
            row[f"{m}|spearman"] = spearmanr(d["transfer_mean"], d[m]).statistic
            row[f"{m}|pearson"] = pearsonr(d["transfer_mean"], d[m]).statistic
    out.append(row)
corr = pd.DataFrame(out).set_index("group")

print("Row unit: one receiver (n=10 teachers each), plus POOLED = the 7 transfer receivers x 10")
print("  teachers = 70 rows. wc1 has no transfer, so its row is n=0 / NaN by construction.")
print("Columns: Spearman / Pearson of transfer_mean (mean test p(dog) over the 56 shared cells,")
print("  per-cell-first) against each screen metric. HEADLINE metric = gated_off_delta.")
print("IMPORTANT: the POOLED row is dominated by the RECEIVER level, not by teacher ranking --")
print("  it mostly re-expresses the between-receiver ordering. The per-receiver rows are the")
print("  informative test of whether the screen RANKS TEACHERS.")
corr.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.6), gridspec_kw={"width_ratios": [1, 1, 0.55]})
MARKS = {"orig": "o", "ds1": "s", "ds2": "D", "ds3": "P", "fs1": "^", "fs2": "v", "fs3": "X"}
COLS = {"orig": "tab:blue", "ds1": "tab:orange", "ds2": "tab:red", "ds3": "tab:brown",
        "fs1": "tab:green", "fs2": "tab:olive", "fs3": "tab:cyan"}
for ax, logy in zip(axes[:2], [False, True]):
    for r in XFER:
        d = df[df["receiver"] == r]
        ax.scatter(d["gated_off_delta"], d["transfer_mean"], s=200, marker=MARKS[r],
                   color=COLS[r], edgecolor="black", linewidth=0.6, label=r, zorder=3)
    for _, row in df[df["receiver"] == "orig"].iterrows():  # annotate orig only (else unreadable)
        ax.annotate(row["teacher"], (row["gated_off_delta"], row["transfer_mean"]),
                    textcoords="offset points", xytext=(7, 4), fontsize=6.5, alpha=0.85)
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("screen: off-topic-GATED peak delta val p(dog)")
    ax.set_ylabel("transfer: mean test p(dog) over 56 shared cells")
    ax.set_title("log y" if logy else "linear y")
    ax.grid(alpha=0.3)
axes[1].legend(title="receiver", loc="center left", bbox_to_anchor=(1.35, 0.5), frameon=False)

wc = df[df["receiver"] == "wc1"]["gated_off_delta"].to_numpy()
axes[2].plot(wc, np.zeros_like(wc), "|", markersize=26, markeredgewidth=2.2, color="tab:purple")
axes[2].set_yticks([])
axes[2].set_ylim(-1, 1)
axes[2].set_xlabel("screen: off-topic-GATED peak delta val p(dog)")
axes[2].set_title("wc1 (no transfer side)\n10 teachers, screen values only")
axes[2].grid(alpha=0.3, axis="x")

fig.suptitle("Row unit = one (receiver, teacher) pair; 70 points in panels 1-2 (7 receivers x 10 "
             "teachers), 10 rug ticks in panel 3.\nx: steerability screen peak delta val p(dog) "
             "restricted to off-topic-gate-passing grid points (favorite-selected, "
             "validation-reported).\ny: mean over the 56 shared student cells of per-cell mean "
             "test p(dog) (ftseeds averaged within cell). Labels = teacher param_tag, orig only.",
             fontsize=9)
fig.tight_layout(rect=(0, 0, 0.9, 0.87))
plt.show()

In [ ]:
# Fixed original-model sender; one point per teacher and receiver.
# Transfer is the mean of per-seed means over the same 56 shared student recipes.
receiver_labels = {
    "orig": "Original (same model)",
    "ds1": "Data seed 1", "ds2": "Data seed 2", "ds3": "Data seed 3",
    "fs1": "Full seed 1", "fs2": "Full seed 2", "fs3": "Full seed 3",
}
fig, ax = plt.subplots(figsize=(8.8, 5.8))
for r in XFER:
    d = df[df["receiver"] == r]
    ax.scatter(d["ungated_delta"], d["transfer_mean"], s=150, marker=MARKS[r],
               color=COLS[r], edgecolor="black", linewidth=0.6,
               label=receiver_labels[r], zorder=3)
ax.set_xlabel("Ungated steering lift, Δp(dog)")
ax.set_ylabel("Mean student p(dog)")
ax.grid(alpha=0.3)
ax.legend(title="Receiver", loc="upper left", bbox_to_anchor=(1.02, 1),
          frameon=False, fontsize=9)
ax.text(1.04, 0.42, "Data seeds:\nsame initialization,\ndifferent data order\n\n"
        "Full seeds:\ndifferent initialization\nand data order",
        transform=ax.transAxes, fontsize=8, va="top", color="0.3")
fig.suptitle("Steering response and transfer vary across receivers", fontsize=12, y=0.98)
fig.text(0.5, 0.925, "Fixed sender: original Pythia-160M dog teachers",
         ha="center", fontsize=10)
fig.subplots_adjust(left=0.10, right=0.66, bottom=0.13, top=0.86)
FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)
# Retain full-precision coordinates and recipe support for future figure-only edits.
df.loc[df["receiver"].isin(XFER),
       ["receiver", "teacher", "ungated_delta", "transfer_mean", "n_cells"]].to_csv(
    FIGDIR / "steerability_ungated_receivers.csv", index=False)
(FIGDIR / "steerability_ungated_receivers_shared_recipes.json").write_text(
    json.dumps(SHARED, indent=2) + "\n")
fig.savefig(FIGDIR / "steerability_vs_transfer_ungated_receivers.png", dpi=150,
            bbox_inches="tight")
fig.savefig(FIGDIR / "steerability_vs_transfer_ungated_receivers.pdf", bbox_inches="tight")
plt.show()


In [ ]:
summ = (df.groupby("receiver")
          .agg(mean_transfer=("transfer_mean", "mean"),
               mean_transfer_x_prior=("transfer_mean_x_prior", "mean"),
               mean_gated_off_delta=("gated_off_delta", "mean"),
               mean_ungated_delta=("ungated_delta", "mean"),
               mean_base_val_p=("base_val_p", "mean"),
               mean_gated_off_n=("gated_off_n", "mean"),
               test_prior=("receiver", lambda s: PRIOR[s.iloc[0]]))
          .loc[list(RECV)])

print("Row unit: one receiver (aggregate over its 10 top-10 kband teachers). 8 rows.")
print("mean_transfer: mean over 10 teachers of the teacher's mean over the 56 shared cells")
print("  (per-cell-first). NaN for wc1 -- no transfer side.")
print("mean_transfer_x_prior: same, in multiples of that receiver base's dog test prior;")
print("  NaN for wc1 (no `weightc-seed1` prior entry in load_priors).")
print("mean_gated_off_delta (HEADLINE) / mean_ungated_delta: mean over teachers of the screen peak.")
print("mean_base_val_p: the receiver base's unsteered validation p(dog) (constant per receiver).")
print("mean_gated_off_n: mean number of grid points surviving the off-topic gate.")
summ.round(4)

In [ ]:
print("CAVEAT CHECK 1 -- teacher validation p(dog) (`teacher_val_p_mean`) saturates, which is why")
print("no teacher-validation correlation column is reported anywhere in this notebook:")
print(df.groupby("teacher")["teacher_val_p"].mean().round(6).to_string())
print()
print("CAVEAT CHECK 2 -- surviving gated grid points per receiver (gated_n):")
print(df.groupby("receiver")[["gated_off_n", "gated_neg_off_n"]].agg(["mean", "min", "max"]).round(2).to_string())

## Section 2 — all dog orig teachers: self screen vs shared-cell transfer

Section 1 asks only whether the screen ranks 10 deliberately-strong teachers. Here we widen the
teacher population to **every dog teacher finetuned on the original pythia-160m that has a
student tree**: each `<tag>_tseed1` directory under
`students/dog/pythia-160m/teacher-pythia-160m/` that contains an `ex10-20_prefix_n100k` subdir.

- **Transfer** = mean over `SHARED2`, the intersection of student-cell tags across **all**
  teachers in this section, of the per-cell mean `sets.test.p_target` (ftseeds averaged within
  cell first). Every teacher is scored on exactly the same cells, so transfer is cell-matched by
  construction; the extra cells that the top-10 teachers carry are simply ignored.
- **Steerability** = the teacher's `steerability/self/aggregate.json` `peaks.gated_off`
  (headline) and `peaks.ungated` (secondary) delta val p(dog).

Self screens for the non-top-10 teachers are being computed as this is written, so the code
reads whatever aggregates exist and reports **N of M**; correlations are over the available
rows only.

In [ ]:
T2ROOT = OUT / "students/dog/pythia-160m/teacher-pythia-160m"
all_teachers = sorted(d.name[: -len("_tseed1")] for d in T2ROOT.iterdir()
                      if d.is_dir() and d.name.endswith("_tseed1") and (d / POOL).is_dir())

CELLS2 = {t: cell_scores(t, "pythia-160m") for t in all_teachers}
CELLS2 = {t: cs for t, cs in CELLS2.items() if cs}
# SHARED2: student cells held by EVERY teacher in this section. Transfer is measured on exactly
# these cells for every teacher; any extra cells a teacher happens to have are ignored.
SHARED2 = sorted(set.intersection(*(set(cs) for cs in CELLS2.values())))
n_shared_cells = len(SHARED2)
for t, cs in CELLS2.items():
    missing = [c for c in SHARED2 if c not in cs]
    assert not missing, (t, missing)
print(f"SHARED2 = {n_shared_cells} student cells held by all {len(CELLS2)} teachers:")
for c in SHARED2:
    print("   ", c)

rows2 = []
for t, cs in CELLS2.items():
    sc = screen_row(t, "self")
    ep = re.search(r"ep(\d+)", t)
    rows2.append({"teacher": t,
                  "transfer_mean": float(np.mean([cs[c] for c in SHARED2])),
                  "epochs": int(ep.group(1)) if ep else np.nan,
                  "top10": t in TAGS, "cells": tuple(sorted(cs)),
                  "has_screen": sc is not None,
                  **(sc or {k: np.nan for k in
                     ["gated_off_delta", "gated_off_n", "ungated_delta", "gated_neg_off_delta",
                      "gated_neg_off_n", "teacher_val_p", "base_val_p", "selection_set"]})})
t2 = pd.DataFrame(rows2)
M, N = len(t2), int(t2["has_screen"].sum())
print(f"\n{N} of {M} teachers have a self screen"
      + (" (screens for the rest are still being computed)." if N < M else "."))
print("Row unit: one dog teacher finetuned on original pythia-160m, teacher ftseed 1.")
print(f"transfer_mean: mean over the {n_shared_cells} SHARED student cells (the same cells for")
print("  every teacher; per-cell-first over ftseeds) of test p(dog).")
t2.loc[t2["has_screen"], ["teacher", "transfer_mean", "epochs", "top10",
                          "gated_off_delta", "ungated_delta", "gated_off_n"]].round(4)


In [ ]:
s2 = t2[t2["has_screen"]].dropna(subset=["transfer_mean", "gated_off_delta"])
out2 = []
for m in ["gated_off_delta", "ungated_delta", "gated_neg_off_delta"]:
    out2.append({"screen_metric": m, "n": len(s2),
                 "spearman": spearmanr(s2["transfer_mean"], s2[m]).statistic,
                 "spearman_p": spearmanr(s2["transfer_mean"], s2[m]).pvalue,
                 "pearson": pearsonr(s2["transfer_mean"], s2[m]).statistic,
                 "pearson_p": pearsonr(s2["transfer_mean"], s2[m]).pvalue})
corr2 = pd.DataFrame(out2).set_index("screen_metric")

print(f"Row unit: one screen metric. n = {len(s2)} teachers (of {M} with student trees) that")
print("  currently have a self screen. Correlations are of transfer_mean (mean over the")
print(f"  {n_shared_cells} shared student cells) against that screen metric, across teachers,")
print("  receiver = orig (self screen) throughout.")
print("HEADLINE metric = gated_off_delta.")
corr2.round(4)

In [ ]:
s2_fav = [json.loads((OUT / "teachers/dog/pythia-160m" / t / "steerability/self/aggregate.json")
                     .read_text())["teacher_pos_p_mean"] for t in s2["teacher"]]
cvals = np.asarray(s2_fav, dtype=float)
cvmin, cvmax = np.percentile(cvals, 5), np.percentile(cvals, 95)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
for ax, logy in zip(axes, [False, True]):
    sc = ax.scatter(s2["gated_off_delta"], s2["transfer_mean"], s=170, c=s2_fav,
                    cmap="viridis", vmin=cvmin, vmax=cvmax,
                    edgecolor="none", zorder=3)
    top = s2[s2["top10"]]
    ax.scatter(top["gated_off_delta"], top["transfer_mean"], s=260, facecolor="none",
               edgecolor="crimson", linewidth=1.8, zorder=4, label="top-10 kband teacher")
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("steer orig->orig (self screen): off-topic-GATED peak delta val p(dog)")
    ax.set_ylabel(f"transfer orig->orig: mean test p(dog) over the {n_shared_cells} "
                  "shared student cells, students on pythia-160m")
    ax.set_title("log y" if logy else "linear y")
    ax.grid(alpha=0.3)
axes[1].legend(loc="center left", bbox_to_anchor=(1.28, 0.6), frameon=False)
cb = fig.colorbar(sc, ax=axes[1], pad=0.16, extend="both")
cb.set_label("teacher favorite-set p(dog) (own preference score), colour clipped to "
             f"5-95th pct [{cvmin:.4f}, {cvmax:.4f}]")
fig.suptitle(f"ORIG -> ORIG. Row unit = one dog orig teacher (ftseed 1); {len(s2)} of {M} teachers with a self "
             "screen today.\nx: self-screen peak delta val p(dog) over off-topic-gate-passing "
             f"grid points. y: mean over the {n_shared_cells} student cells shared by ALL teachers, "
             "of per-cell mean test "
             "p(dog).\nColour = teacher favorite p(dog); red outline = the 10 top-10 kband teachers of "
             "Section 1.", fontsize=9)
fig.tight_layout(rect=(0, 0, 0.88, 0.87))
FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)
fig.savefig(FIGDIR / "steerability_vs_transfer_gatedoff_teachers.png", dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
s2_fav = [json.loads((OUT / "teachers/dog/pythia-160m" / t / "steerability/self/aggregate.json")
                     .read_text())["teacher_pos_p_mean"] for t in s2["teacher"]]
# UNGATED counterpart of the figure above (same panels/markers/legend), x = ungated_delta.
cvals = np.asarray(s2_fav, dtype=float)
cvmin, cvmax = np.percentile(cvals, 5), np.percentile(cvals, 95)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
for ax, logy in zip(axes, [False, True]):
    sc = ax.scatter(s2["ungated_delta"], s2["transfer_mean"], s=170, c=s2_fav,
                    cmap="viridis", vmin=cvmin, vmax=cvmax,
                    edgecolor="none", zorder=3)
    top = s2[s2["top10"]]
    ax.scatter(top["ungated_delta"], top["transfer_mean"], s=260, facecolor="none",
               edgecolor="crimson", linewidth=1.8, zorder=4, label="top-10 kband teacher")
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("steer orig->orig (self screen): UNGATED peak delta val p(dog)")
    ax.set_ylabel(f"transfer orig->orig: mean test p(dog) over the {n_shared_cells} "
                  "shared student cells, students on pythia-160m")
    ax.set_title("ungated, log y" if logy else "ungated, linear y")
    ax.grid(alpha=0.3)
axes[1].legend(loc="center left", bbox_to_anchor=(1.28, 0.6), frameon=False)
cb = fig.colorbar(sc, ax=axes[1], pad=0.16, extend="both")
cb.set_label("teacher favorite-set p(dog) (own preference score), colour clipped to "
             f"5-95th pct [{cvmin:.4f}, {cvmax:.4f}]")
fig.suptitle(f"ORIG -> ORIG. UNGATED variant. Row unit = one dog orig teacher (ftseed 1); {len(s2)} of {M} "
             "teachers with a self screen today.\nx: self-screen peak delta val p(dog) over ALL "
             f"grid points, no off-topic gate. y: mean over the {n_shared_cells} student cells shared "
             "by ALL teachers, of "
             "per-cell mean test p(dog).\nColour = teacher favorite p(dog); red outline = the 10 top-10 "
             "kband teachers of Section 1.", fontsize=9)
fig.tight_layout(rect=(0, 0, 0.88, 0.87))
FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)
fig.savefig(FIGDIR / "steerability_vs_transfer_ungated_teachers.png", dpi=150,
            bbox_inches="tight")
plt.show()


In [ ]:
# Sibling of the figure above with the axes swapped in role: x = the teacher's OWN favorite-set
# p(dog) (the quantity that was only a colour above), colour = the UNGATED steer lift.
cvals = s2["ungated_delta"].to_numpy(dtype=float)
cvmin, cvmax = np.percentile(cvals, 5), np.percentile(cvals, 95)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
for ax, logy in zip(axes, [False, True]):
    sc = ax.scatter(s2_fav, s2["transfer_mean"], s=170, c=s2["ungated_delta"],
                    cmap="viridis", vmin=cvmin, vmax=cvmax, edgecolor="none", zorder=3)
    tm = s2["top10"].to_numpy()
    ax.scatter(np.asarray(s2_fav)[tm], s2["transfer_mean"].to_numpy()[tm], s=260,
               facecolor="none", edgecolor="crimson", linewidth=1.8, zorder=4,
               label="top-10 kband teacher")
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("teacher favorite-set p(dog) (own preference score)")
    ax.set_ylabel(f"transfer orig->orig: mean test p(dog) over the {n_shared_cells} "
                  "shared student cells, students on pythia-160m")
    ax.set_title("log y" if logy else "linear y")
    ax.grid(alpha=0.3)
axes[1].legend(loc="center left", bbox_to_anchor=(1.28, 0.6), frameon=False)
cb = fig.colorbar(sc, ax=axes[1], pad=0.16, extend="both")
cb.set_label("steer orig->orig: UNGATED peak delta val p(dog), colour clipped to "
             f"5-95th pct [{cvmin:.4f}, {cvmax:.4f}]")
fig.suptitle(f"ORIG -> ORIG. Row unit = one dog orig teacher (ftseed 1); {len(s2)} of {M} teachers with a self "
             "screen today.\nx: the teacher's own favorite-set p(dog). y: mean over the "
             f"{n_shared_cells} student cells shared by ALL teachers, of per-cell mean test "
             "p(dog).\nColour = UNGATED self-screen peak delta val p(dog); red outline = the 10 "
             "top-10 kband teachers of Section 1.", fontsize=9)
fig.tight_layout(rect=(0, 0, 0.88, 0.87))
FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)
fig.savefig(FIGDIR / "steerability_vs_transfer_xteacherp_colour_ungated.png", dpi=150,
            bbox_inches="tight")
plt.show()


In [ ]:
import sys
from pathlib import Path
from IPython.display import Image, display

_paper_repo = Path.cwd()
if not (_paper_repo / "scripts").is_dir():
    _paper_repo = _paper_repo.parent
if str(_paper_repo) not in sys.path:
    sys.path.insert(0, str(_paper_repo))
from subliminal_learning.figures.steerability.plot_steerability_transfer import DRAFTS_OUT as PAPER_FIGDIR, load_data

from subliminal_learning.figures.steerability.plot_steerability_transfer import render_original

# Frozen 102-teacher/eight-recipe population; no selected-teacher rings.
render_original(load_data())
display(Image(filename=str(PAPER_FIGDIR / "steerability_logit_student_vs_teacher.png")))


In [ ]:
# Secondary check: hold transfer to ONE of the shared cells (SHARED2), so the teacher-side
# comparison is on a single fixed student recipe rather than a mean over the shared set.
FIXED = sorted(SHARED2)[0]
print(f"FIXED = {FIXED}: the alphabetically-first of the {n_shared_cells} SHARED2 cells "
      f"(held, like all of them, by all {M} teachers).")

fx = []
for _, row in t2[t2["has_screen"]].iterrows():
    v = cell_scores(row["teacher"], "pythia-160m")[FIXED]
    fx.append({"teacher": row["teacher"], "epochs": row["epochs"], "top10": row["top10"],
               "transfer_fixed_cell": v, "gated_off_delta": row["gated_off_delta"],
               "ungated_delta": row["ungated_delta"]})
fxd = pd.DataFrame(fx)
print(f"Row unit: one teacher with a self screen; n = {len(fxd)}.")
print("transfer_fixed_cell: that ONE shared cell's per-cell mean test p(dog) (ftseeds averaged).")
for m in ["gated_off_delta", "ungated_delta"]:
    print(f"  transfer_fixed_cell vs {m}: spearman "
          f"{spearmanr(fxd['transfer_fixed_cell'], fxd[m]).statistic:.3f}, pearson "
          f"{pearsonr(fxd['transfer_fixed_cell'], fxd[m]).statistic:.3f}")

fig, ax = plt.subplots(figsize=(9.5, 5.6))
s = ax.scatter(fxd["transfer_fixed_cell"], fxd["gated_off_delta"], s=170, c=fxd["epochs"],
               cmap="viridis", norm=plt.matplotlib.colors.LogNorm(), zorder=3)
tp = fxd[fxd["top10"]]
ax.scatter(tp["transfer_fixed_cell"], tp["gated_off_delta"], s=260, facecolor="none",
           edgecolor="crimson", linewidth=1.8, zorder=4, label="top-10 kband teacher")
ax.set_xlabel(f"transfer orig->orig on the single shared cell {FIXED}: test p(dog)")
ax.set_ylabel("steer orig->orig (self screen): off-topic-GATED peak delta val p(dog)")
ax.grid(alpha=0.3)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.75), frameon=False)
cb = fig.colorbar(s, ax=ax, pad=0.22)
cb.set_label("teacher epochs")
ax.set_title(f"ORIG -> ORIG. Row unit = one dog orig teacher, ftseed 1, with a self screen "
             f"(n={len(fxd)});\nx = transfer on ONE of the {n_shared_cells} shared student cells "
             f"({FIXED}), y = self-screen gated peak,\ncolour = teacher epochs, red outline = "
             "the 10 top-10 kband teachers of Section 1.", fontsize=9)
fig.tight_layout(rect=(0, 0, 0.82, 1))
plt.show()


In [ ]:
# Partial correlations: does the screen->transfer relation survive holding the teacher's own
# preference score fixed?  Controls are the teacher's own p(dog) on the favorite set
# (`teacher_pos_p_mean`) and on the validation set (`teacher_val_p_mean`), both read from the
# same `steerability/self/aggregate.json` as the screen columns.
def _agg_teacher_ps(tag: str) -> dict:
    d = json.loads((OUT / "teachers/dog/pythia-160m" / tag / "steerability/self/aggregate.json").read_text())
    return {"teacher_fav_p": d["teacher_pos_p_mean"], "teacher_val_p": d["teacher_val_p_mean"]}


from subliminal_learning.stats.core import partial_correlation


def partial_spearman(x, y, z):
    """Partial Spearman via the library (Pearson r of rank residuals; p from t, n-2-k = n-3 dof).
    Tuple adapter: returns (r, p, n)."""
    pc_ = partial_correlation(np.asarray(x, float), np.asarray(y, float), [np.asarray(z, float)])
    return pc_.rho, pc_.p, pc_.n


ps = pd.DataFrame([{"teacher": t, **_agg_teacher_ps(t)} for t in t2["teacher"]])
pc = (t2[["teacher", "transfer_mean", "gated_off_delta", "ungated_delta"]]
      .merge(ps, on="teacher")
      .merge(fxd[["teacher", "transfer_fixed_cell"]], on="teacher")
      .dropna())

print("CONTROL DISTRIBUTIONS (row unit: one dog orig teacher, ftseed 1; n =", len(pc), ")")
for c in ["teacher_fav_p", "teacher_val_p"]:
    v = pc[c]
    print(f"  {c}: min {v.min():.6f}  median {v.median():.6f}  max {v.max():.6f}  "
          f"| >= 0.99: {int((v >= 0.99).sum())} of {len(v)}")

AXES = ["transfer_mean", "transfer_fixed_cell"]
METRICS = ["ungated_delta", "gated_off_delta"]
CTRL = ["teacher_fav_p", "teacher_val_p"]

print("\nWHAT IS BEING PARTIALLED OUT (zero-order Spearman against each control):")
for c in CTRL:
    for v in AXES + METRICS:
        s = spearmanr(pc[v], pc[c])
        print(f"  {v:22s} vs {c}: rho {s.statistic:+.3f}  p {s.pvalue:.2e}")

prows = []
for ax_ in AXES:
    for m in METRICS:
        s0 = spearmanr(pc[ax_], pc[m])
        row = {"transfer_axis": ax_, "steer_metric": m,
               "zero_order_rho": s0.statistic, "zero_order_p": s0.pvalue}
        for c in CTRL:
            r, p, _n = partial_spearman(pc[ax_], pc[m], pc[c])
            row[f"partial|{c.replace('teacher_', '').replace('_p', '')}_rho"] = r
            row[f"partial|{c.replace('teacher_', '').replace('_p', '')}_p"] = p
        row["n"] = len(pc)
        prows.append(row)
partial_tbl = pd.DataFrame(prows).set_index(["transfer_axis", "steer_metric"])

print("\nRow unit: one (transfer axis x steer metric) pair; 2 x 2 = 4 rows, each over the same")
print(f"  n = {len(pc)} dog orig teachers (ftseed 1) that have a self screen and a student tree.")
print(f"  transfer_mean = mean over the {n_shared_cells} SHARED student cells of per-cell mean")
print(f"  test p(dog) (ftseeds averaged first); transfer_fixed_cell = the shared cell {FIXED} only.")
print("Columns: zero_order = plain Spearman of transfer vs steer metric; partial|fav and")
print("  partial|val = partial Spearman controlling for the teacher's own favorite-set / ")
print("  validation-set p(dog) (Pearson r of rank residuals; p from t, n-3 dof).")
print("Aggregation: per-teacher values first, then one correlation across teachers.")
display(partial_tbl.round(4))

SHORT = {"ungated_delta": "ungated steer lift", "gated_off_delta": "off-gated steer lift"}
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
rz = pc["teacher_fav_p"].rank().to_numpy()
A = np.column_stack([np.ones_like(rz), rz])
resid = lambda v: v - A @ np.linalg.lstsq(A, v, rcond=None)[0]  # noqa: E731
for ax, m in zip(axes, METRICS):
    ex = resid(pc[m].rank().to_numpy())
    ey = resid(pc["transfer_mean"].rank().to_numpy())
    r, p, _n = partial_spearman(pc["transfer_mean"], pc[m], pc["teacher_fav_p"])
    ax.scatter(ex, ey, s=170, c=pc["transfer_mean"], cmap="viridis",
               norm=plt.matplotlib.colors.LogNorm(), edgecolor="black", linewidth=0.5,
               zorder=3, label=f"teacher (n={len(pc)})")
    ax.axhline(0, color="grey", lw=0.8)
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_xlabel(f"{SHORT[m]} rank residual")
    ax.set_title(f"partial Spearman rho = {r:+.2f}, p = {p:.1e}")
    ax.grid(alpha=0.3)
axes[0].set_ylabel(f"student p(dog) rank residual ({n_shared_cells} shared cells)")
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
fig.suptitle(f"ORIG -> ORIG, {len(pc)} orig dog teachers, ftseed 1. Rank residuals after "
             "regressing out\nthe rank of the teacher favorite p(dog); colour = raw "
             "transfer_mean (log scale).", fontsize=10)
fig.tight_layout(rect=(0, 0, 0.88, 0.92))
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/steerability_partial_residuals_rank.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Same partial, in RAW units (not ranks): residualise each variable on the teacher's own
# favorite-set p(dog) with a straight-line fit, then plot residual vs residual. The annotated
# r is the partial PEARSON correlation (Pearson r of the two residual vectors).
# Row unit: one dog orig teacher (ftseed 1), n = len(pc); x/y in the variables' native units
# (probabilities), minus the linear teacher-p(dog) fit. r/p from the library partial_correlation
# (method="pearson"); p is on n-2-k = n-3 dof (it was n-2 before 2026-10-07).

def _resid_raw(v, z):
    A = np.column_stack([np.ones(len(z)), np.asarray(z, float)])
    v = np.asarray(v, float)
    return v - A @ np.linalg.lstsq(A, v, rcond=None)[0]


def _partial_pearson_raw(m, mask=None):
    """Partial Pearson r of transfer_mean vs steer metric `m`, controlling for teacher_fav_p,
    in raw probability units. `mask` optionally restricts the teacher subset."""
    d = pc if mask is None else pc[mask]
    ex = _resid_raw(d[m], d["teacher_fav_p"])
    ey = _resid_raw(d["transfer_mean"], d["teacher_fav_p"])
    pc_ = partial_correlation(d["transfer_mean"].to_numpy(float), d[m].to_numpy(float),
                              [d["teacher_fav_p"].to_numpy(float)], method="pearson")
    return pc_.rho, pc_.p, pc_.n, ex, ey


ctrl = "teacher_fav_p"
SHORT = {"ungated_delta": "ungated steer lift", "gated_off_delta": "off-gated steer lift"}
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for ax_, m in zip(axes, METRICS):
    r, p, n, ex, ey = _partial_pearson_raw(m)
    top = pc["teacher"].isin(t2.loc[t2["top10"].astype(bool), "teacher"]).to_numpy()
    ax_.scatter(ex[~top], ey[~top], s=70, alpha=0.75, label="other teachers")
    ax_.scatter(ex[top], ey[top], s=110, facecolors="none", edgecolors="k", linewidths=1.5,
                label="top-10 teachers")
    ax_.axhline(0, color="grey", lw=0.8)
    ax_.axvline(0, color="grey", lw=0.8)
    ax_.set_xlabel(f"{SHORT[m]} residual")
    ax_.set_title(f"partial Pearson r = {r:+.2f}, p = {p:.2g}")
axes[0].set_ylabel(f"student p(dog) residual ({n_shared_cells} shared cells)")
axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
fig.suptitle(f"ORIG -> ORIG, {len(pc)} orig dog teachers, ftseed 1. Residuals after a linear fit "
             "on\nteacher favorite p(dog); raw probability units.", fontsize=10)
fig.tight_layout(rect=(0, 0, 0.88, 0.92))
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/steerability_partial_residuals_raw.png", dpi=150, bbox_inches="tight")
plt.show()

# WHY THE RAW AND RANK PARTIALS DISAGREE, and the same partial off the saturated stripe.
sat = (pc["teacher_fav_p"] >= 0.99).to_numpy()
off = ~sat
print(f"{int(sat.sum())} of {len(pc)} teachers are saturated (teacher favorite p(dog) >= 0.99); "
      f"their ungated\n  steer lift is pinned in "
      f"[{pc.loc[sat, 'ungated_delta'].min():.2f}, {pc.loc[sat, 'ungated_delta'].max():.2f}], so "
      "they share essentially one common x-residual (the\n  vertical stripe in the left panel), "
      f"while their transfer spreads {pc.loc[sat, 'transfer_mean'].min():.2f}-"
      f"{pc.loc[sat, 'transfer_mean'].max():.2f} in raw p(dog).")
print("The raw ungated partial r is carried by that stripe - a saturation / heteroscedasticity")
print("  artefact - whereas the rank partial collapses those ties, which is why the two differ")
print("  in sign. Off-stripe check: same raw partial on the non-saturated teachers only")
print(f"  (teacher favorite p(dog) < 0.99, n = {int(off.sum())}):")
for m in METRICS:
    r_all, p_all, n_all, _, _ = _partial_pearson_raw(m)
    r_off, p_off, n_off, _, _ = _partial_pearson_raw(m, off)
    print(f"  {m:18s} all n={n_all}: r {r_all:+.3f} (p {p_all:.2g})   "
          f"| fav_p<0.99 n={n_off}: r {r_off:+.3f} (p {p_off:.2g})")


## Conclusion

**The off-topic-gated screen separates receivers.** Mean gated peak delta val p(dog) over the
10 teachers is 0.930 (orig), 0.829 (ds2), 0.662 (ds3), 0.511 (ds1) — then a cliff — 0.006 (fs3),
0.0044 (fs1), 0.0031 (wc1), 0.0012 (fs2). Mean transfer over the 56 shared cells follows the same
two-tier split: 0.605 (orig), 0.058 (ds2), 0.042 (ds1), 0.041 (ds3) versus 0.019 (fs3), 0.0093
(fs1), 0.0070 (fs2), i.e. 62.9x / 5.3x / 3.1x / 4.5x the base dog test prior for the data-seed
lineages against 1.7x / 1.1x / 0.8x for the fully re-seeded ones. So the gated screen calls the
*steerable* lineages (orig, ds*) apart from the *unsteerable* ones (fs*, wc1) exactly as transfer
does, and wc1's screen values (0.0022–0.0048) place it with the fs receivers — a prediction we
cannot check, since wc1 has no transfer side. Within the steerable tier the ordering is not
reproduced: the screen ranks ds2 > ds3 > ds1 while transfer ranks ds2 > ds1 ~ ds3, and orig's
screen advantage over ds2 (0.930 vs 0.829, a factor 1.12) is nothing like its transfer advantage
(0.605 vs 0.058, a factor 10.5). The pooled 70-row correlation (Spearman 0.874, Pearson 0.603 on
the gated metric) is a restatement of the two-tier receiver split, not evidence about teachers.

**Within a receiver (n=10) the screen does not rank teachers.** Gated Spearman is -0.297 (orig),
0.442 (ds1), 0.115 (ds2), -0.539 (ds3), -0.079 (fs1), -0.212 (fs2), -0.285 (fs3): five of seven
negative, none near the |rho| ~ 0.65 needed for p=0.05 at n=10, and the sign is unstable across
lineages that differ only in a training seed. The ungated column is no better (0.406 / 0.442 /
-0.103 / -0.382 / -0.079 / -0.212 / -0.285) and gated_neg_off is worse still. On the three fs
receivers the gated, ungated and gated_neg_off columns are numerically identical, because at
delta ~ 0.005 the peak grid point passes every gate — there is nothing to gate.

**Across the wide teacher population the screen does rank teachers, moderately.** All 102
teachers are screened, and transfer — mean over the 8 student cells shared by every one of them,
so the comparison is cell-matched by construction — correlates with the gated delta at Spearman
0.640 (p=4e-13) and Pearson 0.552; the ungated delta does better still, Spearman 0.884 (p<1e-34),
Pearson 0.653, while gated_neg_off is flat (0.035, p=0.73). Holding transfer to a single one of
those shared cells — `ep10_lr1e-4_r8a32`, the alphabetically-first — gives Spearman 0.600 gated /
0.855 ungated, i.e. the same picture. The reason Section 1 sees nothing and Section 2 sees a lot
is range: the top-10 kband teachers all sit at gated delta 0.88–0.98, whereas the full population
runs from 0.40 (ep1_lr5e-4_r8a32) to 0.98, and the weak teachers at the bottom of that range are
exactly the weak distillers (ep1 teachers: transfer 0.016–0.38).

**Holding the teacher's own preference score fixed kills the teacher-level relation.** The
teacher's own favorite-set p(dog) (`teacher_pos_p_mean`) is itself saturated for most of the
population (median 0.9967, 61 of 102 at >= 0.99, but a long lower tail down to 0.340), and it
tracks both sides of the correlation almost perfectly: Spearman 0.939 with shared-cell transfer and
0.950 with the ungated delta (0.651 with the gated delta). Partialling it out on the ranks leaves
nothing: transfer_mean vs ungated_delta falls from 0.884 to -0.075 (p=0.45) and vs gated_off_delta
from 0.640 to 0.110 (p=0.27); on the shared cell `ep10_lr1e-4_r8a32` it falls from 0.855 to -0.085
and from 0.600 to 0.023. Controlling for the validation-set score instead gives the same picture
(-0.151 / 0.110 / -0.176 / 0.007, all p > 0.07). So the Section-2 "screen ranks teachers" result
is not screen-specific information: both the screen and distillation transfer are reading the same
underlying quantity, how strongly the teacher itself prefers dog, and the screen adds no ranking
signal once that is held fixed.

**Caveats.**
- *Teacher validation score is useless as a predictor and is not reported.* `teacher_val_p_mean`
  is 0.99956–0.99999 for all 10 Section-1 teachers — saturated — so a correlation column against
  it would be noise on a constant. It has been dropped everywhere.
- *Section 2 transfer is restricted to 8 shared cells.* Every Section-2 teacher is scored on the
  same 8 student cells (the intersection over all 102); the extra cells the top-10 teachers carry
  (63, 73 or 518 each) are ignored, so no teacher is advantaged by its cell mix — but the 8 cells
  are a narrow, low-rank slice of the student grid, not a representative sample of it.
- *Single teacher seed.* Every point is teacher ftseed 1; within-receiver teacher differences are
  not separated from teacher-seed noise.
- *The gated peaks are single grid points.* `gated_off_n` = 1 (min = max = 1) for all
  (receiver, teacher) pairs, so the "gated peak" is one surviving (layer, alpha) reading rather
  than a peak over a set.
- *wc1 has no transfer side and no prior.* Its student tree contains none of the 56 shared cells,
  and `load_priors` has no `weightc-seed1` entry, so both its transfer and x-prior columns are
  NaN by construction; it appears only as the screen-value rug in the third scatter panel.
- *Prior reference.* Transfer multiples use each receiver base's own dog `test_prior`; the
  screen's `base_val_p` is a different prompt set (`validation`) and the two must not be mixed.

**Bottom line.** The off-topic-gated steerability screen is a good cheap classifier of *which
receiver lineage* can be subliminally taught and a moderate cheap ranker of *which teacher*
teaches best when the teacher population spans a real range of strengths — but it carries no
usable signal inside an already-strong, narrow teacher slice.

The raw-unit and rank partials disagree by construction, not by substance: the 61 saturated teachers (favorite p(dog) >= 0.99), whose ungated steer lift is pinned near its ceiling, share a single common x-residual (the vertical stripe in the raw figure) while their transfer residuals spread over 0-0.4, so the positive raw partial (+0.31) is carried entirely by that stripe -- a saturation / heteroscedasticity artefact -- while the rank partial (-0.08) collapses the ties and shows no relation. The off-stripe partials, restricted to the non-saturated teachers with favorite p(dog) < 0.99, are printed with the raw figure.


## Section 3 — teacher Pile damage (dCE, KL) as the predictor, same-model vs cross-model

Sections 1–2 used the steerability screen as the candidate predictor and found that it adds
nothing once the teacher's own preference score is held fixed. Here the candidate predictor is
instead the teacher's **Pile damage**: `delta_ce` (student CE minus base CE on the pp1 Pile probe)
and `kl_base_student`, read from
`teachers/dog/pythia-160m/{tag}/ftseed1/eval/pile_probe.json` (`pool_version == "pp1"` asserted
for all 102 teachers).

- **Same-model (ORIG -> ORIG)** transfer: students finetuned on the *original* pythia-160m, i.e.
  the same base the teacher was finetuned from.
- **Cross-model (ORIG -> DS1)** transfer: students on `pythia-160m-data-seed1`, a different
  pretraining data order — the receiver never shares the teacher's base weights.
- Both receivers are scored on **exactly the same 8 shared cells** (`SHARED2`, the cell
  intersection over all 102 teachers of Section 2), per-cell-first over ftseeds; every teacher is
  asserted to hold all 8 on both receivers. Restricting to shared cells is what makes teachers
  comparable — no teacher is advantaged by its cell mix.
- **Control**: logit of the teacher's own favorite-set p(dog) (`teacher_pos_p_mean`), the same
  control as Section 2's partials. **Outcome**: logit of the student p(dog) transfer mean
  (p clipped to [1e-6, 1-1e-6]). The steer columns are carried along as comparison predictors.


In [ ]:
# Section 3 frame: teacher Pile damage (pp1 probe) + both receivers' shared-cell transfer.
EPS3 = 1e-6


def _logit3(p):
    p = np.clip(np.asarray(p, dtype=float), EPS3, 1.0 - EPS3)
    return np.log(p / (1.0 - p))


def _pile_probe(tag: str) -> dict:
    d = json.loads((OUT / "teachers/dog/pythia-160m" / tag / "ftseed1/eval/pile_probe.json").read_text())
    assert d["pool_version"] == "pp1", (tag, d["pool_version"])
    return {"dce": float(d["delta_ce"]), "kl": float(d["kl_base_student"])}


CELLS2_DS1 = {t: cell_scores(t, "pythia-160m-data-seed1") for t in CELLS2}
for t in CELLS2:  # both receivers must hold every shared cell for every teacher
    for lbl, cs in (("orig", CELLS2[t]), ("ds1", CELLS2_DS1[t])):
        missing = [c for c in SHARED2 if c not in cs]
        assert not missing, (lbl, t, missing)

rows3 = []
for t in CELLS2:
    agg = json.loads((OUT / "teachers/dog/pythia-160m" / t / "steerability/self/aggregate.json").read_text())
    rows3.append({"teacher": t, **_pile_probe(t),
                  "teacher_fav_p": agg["teacher_pos_p_mean"],
                  "steer_ungated": agg["peaks"]["ungated"]["delta_val_p_mean"],
                  "steer_gated_off": agg["peaks"]["gated_off"]["delta_val_p_mean"],
                  "top10": t in TAGS,
                  "tr_orig": float(np.mean([CELLS2[t][c] for c in SHARED2])),
                  "tr_ds1": float(np.mean([CELLS2_DS1[t][c] for c in SHARED2]))})
s3 = pd.DataFrame(rows3)
s3["log_dce"] = np.log(s3["dce"])
s3["log_kl"] = np.log(s3["kl"])
s3["ctrl"] = _logit3(s3["teacher_fav_p"])
s3["y_orig"] = _logit3(s3["tr_orig"])
s3["y_ds1"] = _logit3(s3["tr_ds1"])
print(f"Row unit: one dog teacher finetuned on original pythia-160m, ftseed 1; n = {len(s3)}.")
print(f"Transfer = mean over the {n_shared_cells} SHARED2 cells of per-cell mean test p(dog)")
print("  (ftseeds averaged inside a cell first), on receiver orig (same-model) and data-seed1")
print("  (cross-model). Pile damage from the teacher's own pp1 pile_probe.json.")
s3[["teacher", "dce", "kl", "teacher_fav_p", "steer_ungated", "steer_gated_off",
    "tr_orig", "tr_ds1", "top10"]].round(4)

In [ ]:
# Table: rank / raw zero-order, partial and semipartial correlations of each predictor with
# logit transfer, controlling for logit teacher favorite p(dog). One row per (receiver, predictor).
def _resid3(v, z):
    A = np.column_stack([np.ones(len(z)), np.asarray(z, float)])
    v = np.asarray(v, float)
    return v - A @ np.linalg.lstsq(A, v, rcond=None)[0]


def _corrs3(x, y, z, rank: bool) -> dict:
    """Zero-order, partial and semipartial (x residualised, y raw) correlation of x with y given z.
    rank=True runs all three on the ranks; p is two-sided from t with n-3 dof (partial)."""
    if rank:
        x, y, z = (pd.Series(np.asarray(v, float)).rank().to_numpy() for v in (x, y, z))
    x, y, z = (np.asarray(v, float) for v in (x, y, z))
    ex = _resid3(x, z)
    r0 = float(np.corrcoef(x, y)[0, 1])
    pc_ = partial_correlation(x, y, [z], method="pearson")  # already ranked when rank=True
    rsp = float(np.corrcoef(ex, y)[0, 1])
    return {"r0": r0, "partial": pc_.rho, "p": pc_.p, "semip": rsp}


PRED3 = {"log dCE": "log_dce", "log KL": "log_kl",
         "steer ungated": "steer_ungated", "steer gated_off": "steer_gated_off"}
RECV3 = {"orig (same-model)": "y_orig", "ds1 (cross-model)": "y_ds1"}
rows = []
for rlab, ycol in RECV3.items():
    for plab, pcol in PRED3.items():
        rk = _corrs3(s3[pcol], s3[ycol], s3["ctrl"], rank=True)
        rw = _corrs3(s3[pcol], s3[ycol], s3["ctrl"], rank=False)
        rows.append({"receiver": rlab, "predictor": plab,
                     "rank_r0": rk["r0"], "rank_partial": rk["partial"], "rank_partial_p": rk["p"],
                     "rank_semipartial": rk["semip"],
                     "raw_r0": rw["r0"], "raw_partial": rw["partial"], "raw_partial_p": rw["p"],
                     "raw_semipartial": rw["semip"]})
tbl3 = pd.DataFrame(rows).set_index(["receiver", "predictor"])

print("Row unit: one (receiver x predictor) pair; 2 receivers x 4 predictors = 8 rows, each over")
print(f"  the same n = {len(s3)} dog orig teachers (ftseed 1).")
print("Selection: all teachers with a student tree on BOTH receivers; transfer is the mean over")
print(f"  the {n_shared_cells} SHARED2 cells of per-cell mean test p(dog) (per-cell-first).")
print("Outcome = logit(transfer p(dog)); control = logit(teacher favorite p(dog)).")
print("Columns: rank_* = correlations computed on ranks (Spearman family), raw_* = on the values")
print("  as given (log dCE / log KL / steer deltas vs logit transfer). r0 = zero-order;")
print("  partial = both sides residualised on the control; semipartial = predictor residualised,")
print("  outcome raw (its square = incremental variance explained); p from t with n-3 dof.")
display(tbl3.round(3))

print("\nCOLLINEARITY (Spearman rho, same n):")
for a, b in [("dce", "teacher_fav_p"), ("kl", "teacher_fav_p"), ("dce", "kl"),
             ("steer_ungated", "teacher_fav_p"), ("tr_orig", "tr_ds1")]:
    print(f"  {a:14s} ~ {b:14s}: {spearmanr(s3[a], s3[b]).statistic:+.3f}")

In [ ]:
# Figure A: teacher Pile dCE (log x) vs student p(dog) (logit y) on both receivers.
cvals3 = s3["teacher_fav_p"].to_numpy(dtype=float)
cvmin3, cvmax3 = np.percentile(cvals3, 5), np.percentile(cvals3, 95)
P_TICKS = [0.01, 0.1, 0.5, 0.9, 0.99, 0.999, 0.9999]
tm3 = s3["top10"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8), sharex=True)
for ax, (rlab, ycol) in zip(axes, RECV3.items()):
    y = s3[ycol].to_numpy()
    sc = ax.scatter(s3["dce"], y, s=150, c=cvals3, cmap="viridis", vmin=cvmin3, vmax=cvmax3,
                    edgecolor="none", zorder=3, label=f"teacher (n={len(s3)})")
    ax.scatter(s3["dce"].to_numpy()[tm3], y[tm3], s=240, facecolor="none", edgecolor="crimson",
               linewidth=1.8, zorder=4, label="top-10 kband teacher")
    ax.set_xscale("log")
    keep = [(p, _logit3(p)) for p in P_TICKS if y.min() - 0.5 <= _logit3(p) <= y.max() + 0.5]
    ax.set_yticks([v for _, v in keep])
    ax.set_yticklabels(["%g" % p for p, _ in keep])
    rk = _corrs3(s3["log_dce"], s3[ycol], s3["ctrl"], rank=True)
    ax.set_title(f"ORIG -> {rlab.split(' ')[0].upper()}   rank r0 = {rk['r0']:+.2f}, "
                 f"partial | teacher p = {rk['partial']:+.2f}", fontsize=10)
    ax.set_xlabel("teacher Pile damage: delta_ce (pp1 probe), log axis")
    ax.grid(alpha=0.3)
axes[0].set_ylabel(f"student p(dog), mean over the {n_shared_cells} shared cells\n"
                   "(logit axis, ticks labelled with p)")
axes[1].legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False)
fig.suptitle(f"Row unit = one dog orig teacher, ftseed 1 (n={len(s3)}); both panels use the same "
             f"{n_shared_cells} SHARED2 student cells,\nper-cell-first over ftseeds. x = the "
             "teacher's own pp1 Pile delta_ce; y = logit student p(dog).\nColour = teacher "
             "favorite p(dog); red outline = the 10 top-10 kband teachers of Section 1.",
             fontsize=9)
fig.tight_layout(rect=(0, 0, 0.84, 0.86))
cb = fig.colorbar(sc, ax=axes, fraction=0.03, pad=0.16, extend="both")
cb.set_label("teacher favorite-set p(dog), colour clipped to "
             f"5-95th pct [{cvmin3:.4f}, {cvmax3:.4f}]", fontsize=8)
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/teacher_dce_vs_transfer_orig_ds1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Figure B: added-variable (partial-regression) plots for log dCE on both receivers. Both axes
# are residualised on the logit teacher favorite p(dog); the slope is the partial relation.
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for ax_, (rlab, ycol) in zip(axes, RECV3.items()):
    ex = _resid3(s3["log_dce"], s3["ctrl"])
    ey = _resid3(s3[ycol], s3["ctrl"])
    rw = _corrs3(s3["log_dce"], s3[ycol], s3["ctrl"], rank=False)
    ax_.scatter(ex[~tm3], ey[~tm3], s=70, alpha=0.75, label="other teachers")
    ax_.scatter(ex[tm3], ey[tm3], s=110, facecolors="none", edgecolors="k", linewidths=1.5,
                label="top-10 teachers")
    ax_.axhline(0, color="grey", lw=0.8)
    ax_.axvline(0, color="grey", lw=0.8)
    ax_.set_xlabel("log teacher dCE residual")
    ax_.set_title(f"ORIG -> {rlab.split(' ')[0].upper()}: partial r = {rw['partial']:+.2f}, "
                  f"p = {rw['p']:.2g}", fontsize=10)
    ax_.grid(alpha=0.3)
axes[0].set_ylabel(f"logit student p(dog) residual ({n_shared_cells} shared cells)")
axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
fig.suptitle(f"Added-variable plots, {len(s3)} orig dog teachers, ftseed 1. Both axes residualised "
             "on\nlogit teacher favorite p(dog); same SHARED2 cells on both receivers.", fontsize=10)
fig.tight_layout(rect=(0, 0, 0.88, 0.92))
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/teacher_dce_partial_residuals_orig_ds1.png", dpi=150, bbox_inches="tight")
plt.show()

### Section 3 conclusion — Pile damage, unlike steerability, keeps a conditional relation with same-model transfer

**Same-model (ORIG -> ORIG).** Teacher Pile damage is the strongest zero-order predictor in this
notebook (rank rho = 0.95 for log dCE) and, unlike the steerability screen, it *survives* the
control: partialling out the teacher's own favorite p(dog) leaves rank partial rho = 0.73
(p ~ 1e-17). The rank semipartial is 0.25, i.e. log dCE adds roughly 6% of the outcome variance on
top of the teacher preference score — a real but modest increment, because the control is itself
strongly collinear with the predictor (Spearman dCE ~ favorite p = 0.89).

**Cross-model (ORIG -> DS1).** The same predictor gives rank rho = 0.67 zero-order, falling to a
partial of 0.27 (semipartial 0.21). This is an **upper bound**: with off-topic p added as a second
control, the audit notebook §2d takes the cross-model partial to ~0. So the honest reading is that
Pile damage predicts cross-model transfer weakly at best.

**Steerability keeps nothing on either receiver.** The ungated steer lift goes 0.88 -> -0.08
(same-model) and 0.60 -> -0.10 (cross-model) under the same control, and the off-topic-gated lift
likewise (0.64 -> 0.11, 0.42 -> -0.01); all four partials have p > 0.27. This reproduces Section 2's
finding on a second receiver: the screen is a proxy for the teacher's own preference, nothing more.

**dCE and KL are interchangeable.** Spearman dCE ~ KL = 1.00 across the 102 teachers, and their
rows in the table agree to two decimals throughout. Either can be quoted; nothing distinguishes
the CE and KL readings of the pp1 probe at this population's damage range.

**Caveat — the two outcomes are not independent.** The same-model and cross-model transfer means
correlate at Spearman 0.70 over these teachers. A predictor that genuinely relates only to
same-model transfer will therefore still show a positive cross-model number; the cross-model
partial of 0.27 is consistent with pure leakage from the same-model relation, which is why it is
reported as an upper bound rather than as evidence of a cross-model effect.


In [ ]:
import sys
from pathlib import Path
from IPython.display import Image, display

_paper_repo = Path.cwd()
if not (_paper_repo / "scripts").is_dir():
    _paper_repo = _paper_repo.parent
if str(_paper_repo) not in sys.path:
    sys.path.insert(0, str(_paper_repo))
from subliminal_learning.figures.steerability.plot_steerability_transfer import DRAFTS_OUT, STEERABILITY_OUT, load_data

from subliminal_learning.figures.steerability.plot_steerability_transfer import render_combined, render_same_cross

# Rank first, then residualize both variables on ranked teacher preference.
# Nominal partial-correlation tests use n - 1 control - 2 degrees of freedom.
_paper_data = load_data()
render_combined(_paper_data)
render_same_cross(_paper_data)
for _paper_dir, _paper_stem in ((DRAFTS_OUT, "steerability_teacher_strength_adjusted"),
                                 (STEERABILITY_OUT, "same_vs_cross_transfer_teacher_adjusted")):
    display(Image(filename=str(_paper_dir / f"{_paper_stem}.png")))
